In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm

from src.data import get_electrode_df, add_metadata_features
from src.models.decoding import run_decoding_population, get_population_ensemble_predictions

In [ ]:
all_epochs = list(Path("outputs/epochs_preprocessed").glob("*_epo.fif"))
all_results = list(Path("outputs/causal4/find_As").glob("*_results.csv"))
all_decoders = list(Path("outputs/causal4/find_As").glob("*_decoders.csv"))
all_electrode_dfs = list(Path("outputs/causal4/find_speech_responsive").glob("*_results.csv"))

window_size = 0.3
pca_num_components = 0.5

outdir = "."

In [ ]:
epochs = {re.findall(r"(EC[\d]+)_epo", str(path))[0]: mne.read_epochs(path)
          for path in all_epochs}
for ep in epochs.values():
    ep.metadata = add_metadata_features(ep.metadata)

In [ ]:
# Map Freesurfer ROIs to coarser ROIs
roi_map = {
    "parstriangularis": "ifg",
    "parsopercularis": "ifg",
}

In [ ]:
results = pd.concat([pd.read_csv(path) for path in all_results])

# merge in electrode metadata
electrode_df = pd.concat([pd.read_csv(path) for path in all_electrode_dfs]).set_index(["subject", "electrode_idx"])

results = results.merge(electrode_df, on=["subject", "electrode_idx"], how="left")
results["coarse_roi"] = results.roi.map(roi_map).fillna(results.roi)

A_sites = results[results.A].set_index(["subject", "electrode_idx", "phoneme_pair", "smin", "smax"])
A_sites

## Learn coarse-ROI-level population decoders

In [ ]:
# # Manually pick the PCA num components based on held out dev generalization.

# def run_roi_decoders(pca_num_components=0.9):
#     all_test_scores = []

#     for (subject, roi, phoneme_pair), sites in tqdm(A_sites.groupby(["subject", "coarse_roi", "phoneme_pair"])):
#         electrode_idxs = sorted(sites.index.get_level_values("electrode_idx").unique())

#         epochs_i = epochs[subject]
#         md_i = epochs_i.metadata
#         epochs_i = epochs_i[md_i[md_i.resampled.isin((1, 6))].index]

#         global_min_sample = epochs_i.times.tolist().index(0)
#         global_max_sample = global_min_sample + int(window_size * epochs_i.info['sfreq'])

#         train_scores, test_scores, outcomes, models = run_decoding_population(
#             epochs_i=epochs_i,
#             electrode_idxs=electrode_idxs,
#             phoneme_pair=phoneme_pair,
#             subject=subject,
#             population_name=roi,
#             window_size=int(window_size * epochs_i.info["sfreq"]),
#             stride=5,
#             global_min_sample=global_min_sample,
#             global_max_sample=global_max_sample,
#             target="acoustic",
#             strategy="train-test",
#             pca_num_components=pca_num_components,
#         )
#         scores_df = pd.concat(
#             {key: pd.DataFrame(scores_i) for key, scores_i in test_scores.items()},
#             names=["smin", "smax", "fold"]).assign(
#                 subject=subject,
#                 coarse_roi=roi,
#                 phoneme_pair=phoneme_pair,
#                 num_electrodes=len(electrode_idxs),
#             )
#         all_test_scores.append(scores_df)
        
#     # Group all_test_scores and results by subject, coarse_roi, and phoneme_pair
#     roi_level_scores = pd.concat(all_test_scores).reset_index()
#     electrode_level_scores = results

#     summary = []
#     for (subject, roi, phoneme_pair), roi_scores in roi_level_scores.groupby(['subject', 'coarse_roi', 'phoneme_pair']):
#         # Exclude single-electrode ROIs for this analysis asking whether ROIs are better than singles
#         num_electrodes = roi_scores['num_electrodes'].iloc[0]
#         if num_electrodes < 2:
#             continue

#         # Get mean held-out (test) performance for this ROI
#         roi_mean_auc = roi_scores['roc_auc'].mean()
        
#         # Get constituent electrodes for this ROI/subject/phoneme_pair
#         mask = (
#             (electrode_level_scores['subject'] == subject) &
#             (electrode_level_scores['coarse_roi'] == roi) &
#             (electrode_level_scores['phoneme_pair'] == phoneme_pair)
#         )
#         electrodes_auc = electrode_level_scores.loc[mask, 'roc_auc']
#         if len(electrodes_auc) == 0:
#             continue  # skip if no electrodes
        
#         # Compare ROI mean to best electrode
#         best_electrode_auc = electrodes_auc.max()
#         summary.append({
#             'subject': subject,
#             'coarse_roi': roi,
#             'phoneme_pair': phoneme_pair,
#             'roi_mean_auc': roi_mean_auc,
#             'best_electrode_auc': best_electrode_auc,
#             'roi_better_than_best_electrode': roi_mean_auc > best_electrode_auc
#         })

#     summary_df = pd.DataFrame(summary)
#     return summary_df
    

# summary_dfs = {}
# for pca_num_components in tqdm(np.linspace(0.4, 0.95, 10, endpoint=False)):
#     print(f"Running PCA with {pca_num_components} components")
#     summary_df = run_roi_decoders(pca_num_components=pca_num_components)
#     summary_dfs[pca_num_components] = summary_df

# sdf = pd.concat(summary_dfs, names=["num_components"]).droplevel(-1).reset_index()
# sdf["diff"] = sdf.roi_mean_auc - sdf.best_electrode_auc
# sdf.groupby("num_components")["diff"].agg(["mean", "max", "min", "median"])

In [ ]:
all_train_scores, all_test_scores = {}, {}
all_outcomes, all_held_out_outcomes, all_models = {}, {}, {}
all_populations = {}
for (subject, roi, phoneme_pair), sites in tqdm(A_sites.groupby(["subject", "coarse_roi", "phoneme_pair"])):
    electrode_idxs = sorted(sites.index.get_level_values("electrode_idx").unique())

    epochs_i = epochs[subject]
    md_i = epochs_i.metadata
    held_out_epochs = epochs_i[md_i[~md_i.resampled.isin((1, 6))].index]
    epochs_i = epochs_i[md_i[md_i.resampled.isin((1, 6))].index]
    
    global_min_sample = epochs_i.times.tolist().index(0)
    global_max_sample = global_min_sample + int(window_size * epochs_i.info['sfreq'])

    train_scores, test_scores, outcomes, models = run_decoding_population(
        epochs_i=epochs_i,
        electrode_idxs=electrode_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name=roi,
        window_size=int(window_size * epochs_i.info["sfreq"]),
        stride=5,
        global_min_sample=global_min_sample,
        global_max_sample=global_max_sample,
        target="acoustic",
        strategy="train-test",
        pca_num_components=pca_num_components,
    )

    score_meta = {"num_electrodes": len(electrode_idxs)}

    held_out_outcomes = {
        model_key: get_population_ensemble_predictions(
            model_key, models_i,
            electrode_idxs=electrode_idxs,
            epochs=held_out_epochs
        )
        for model_key, models_i in models.items()
    }

    train_scores_df = pd.concat(
        {key: pd.DataFrame(scores_i) for key, scores_i in train_scores.items()},
        names=["subject", "population_name", "phoneme_pair", "smin", "smax", "fold"]) \
            .assign(**score_meta)
    scores_df = pd.concat(
        {key: pd.DataFrame(scores_i) for key, scores_i in test_scores.items()},
        names=["subject", "population_name", "phoneme_pair", "smin", "smax", "fold"]) \
            .assign(**score_meta)

    all_train_scores[(subject, roi, phoneme_pair)] = train_scores_df
    all_test_scores[(subject, roi, phoneme_pair)] = scores_df
    all_outcomes[(subject, roi, phoneme_pair)] = outcomes
    all_held_out_outcomes[(subject, roi, phoneme_pair)] = held_out_outcomes
    all_models[(subject, roi, phoneme_pair)] = models
    all_populations[(subject, roi, phoneme_pair)] = electrode_idxs

In [ ]:
# Group all_test_scores and results by subject, population_name, and phoneme_pair
roi_level_scores = pd.concat(all_test_scores).reset_index()
electrode_level_scores = results

summary = []
for (subject, roi, phoneme_pair), roi_scores in roi_level_scores.groupby(['subject', 'population_name', 'phoneme_pair']):
    # Exclude single-electrode ROIs for this analysis asking whether ROIs are better than singles
    num_electrodes = roi_scores['num_electrodes'].iloc[0]
    if num_electrodes < 2:
        continue

    # Get mean held-out (test) performance for this ROI
    roi_mean_auc = roi_scores['roc_auc'].mean()
    
    # Get constituent electrodes for this ROI/subject/phoneme_pair
    mask = (
        (electrode_level_scores['subject'] == subject) &
        (electrode_level_scores['coarse_roi'] == roi) &
        (electrode_level_scores['phoneme_pair'] == phoneme_pair)
    )
    electrodes_auc = electrode_level_scores.loc[mask, 'roc_auc']
    if len(electrodes_auc) == 0:
        continue  # skip if no electrodes
    
    # Compare ROI mean to best electrode
    best_electrode_auc = electrodes_auc.max()
    summary.append({
        'subject': subject,
        'population_name': roi,
        'phoneme_pair': phoneme_pair,
        'roi_mean_auc': roi_mean_auc,
        'best_electrode_auc': best_electrode_auc,
        'roi_better_than_best_electrode': roi_mean_auc > best_electrode_auc
    })

summary_df = pd.DataFrame(summary)
summary_df["roi_diff"] = summary_df.roi_mean_auc - summary_df.best_electrode_auc

In [ ]:
summary_df.roi_better_than_best_electrode.mean()

In [ ]:
ax = sns.boxplot(data=summary_df, x="roi_diff", hue="phoneme_pair")
ax.axvline(0, color='k', linestyle='--')

In [ ]:
ax = sns.boxplot(data=summary_df, x="roi_diff", hue="population_name")
ax.axvline(0, color='k', linestyle='--')

In [ ]:
# there should be just one model ensemble per population
for _, models in all_models.items():
    assert len(models) == 1

# OK, simplify representations then
all_models = {population_key: next(iter(model_list.values()))
              for population_key, model_list in all_models.items()}
all_outcomes = {population_key: next(iter(outcomes_list.values()))
                for population_key, outcomes_list in all_outcomes.items()}
all_held_out_outcomes = {population_key: next(iter(held_out_list.values()))
                         for population_key, held_out_list in all_held_out_outcomes.items()}

all_train_scores_df = pd.concat(all_train_scores.values()).reset_index()
all_test_scores_df = pd.concat(all_test_scores.values()).reset_index()

In [ ]:
avg_scores_df = pd.concat(all_test_scores.values()).groupby(["subject", "population_name", "phoneme_pair", "smin", "smax", "num_electrodes"]) \
    .mean().sort_values("roc_auc", ascending=False)
avg_scores_df

## Estimate significance threshold

The expected ROC-AUC for a random classifier is distributed $\text{Beta}(N_+, N_-)$, where $N_+$ is the number of positive samples and $N_-$ is the number of negative samples. We can use this to estimate a significance threshold for our ROC-AUC scores.

In [ ]:
data_sample = next(iter(all_outcomes.values())).query("fold == 0")  # don't over-estimate sample counts by double-counting across folds
class_counts = data_sample.groupby("decoder_target").size().tolist()
print(f"Class counts: {class_counts}")

from scipy.stats import beta
confidence_threshold = 0.95
alpha = 1 - confidence_threshold
significance_threshold = beta.ppf(1 - alpha, *class_counts)
print(f"Significance threshold: {significance_threshold:.3f}")

In [ ]:
import seaborn as sns
g = sns.displot(data=avg_scores_df,
                x="roc_auc", aspect=2, height=4, kind="kde",
                clip=(0, 1), fill=True, color="blue")
g.ax.axvline(significance_threshold, color="red", linestyle="--", label="Significance threshold")

## Save results

In [ ]:
avg_scores_df = avg_scores_df[avg_scores_df.roc_auc >= significance_threshold]
avg_scores_df.to_csv(f"{outdir}/results.csv")

In [ ]:
torch.save({
    "train_scores": all_train_scores_df,
    "test_scores": all_test_scores_df,
    "avg_scores": avg_scores_df,
    "outcomes": all_outcomes,
    "held_out_outcomes": all_held_out_outcomes,
    "models": all_models,
    "populations": all_populations,
}, f"{outdir}/unified_decoders.pt")